# 3 — Gradient descent and SGD by hand

In this notebook we will:

1. Recap **gradient descent** (GD) as the standard training algorithm.
2. Apply GD to a univariate function to study how the **learning rate** $\alpha$ controls convergence.
3. Set up a **two-parameter** linear regression problem and derive its gradient symbolically — by hand.
4. Implement plain **GD** and **mini-batch SGD** as `numpy` for-loops and analyse their convergence behaviour.

We do not use `torch.nn`, `.backward()`, or autograd. Every gradient in this notebook is one you derive; every parameter update is one you write.

Suggested reading: `lectures/4-1847-1960-GD+SGD.pdf` introduces every formula re-derived here.

## Recap — the training problem

Once a model architecture is fixed, training amounts to assigning numerical values to its parameters. Given a training set $\{(x_i, y_i)\}_{i \in [k]}$ and a loss function $L$, training is the **mathematical optimisation problem**

$$ (w^*,\, b^*) \;\in\; \operatorname*{argmin}_{w,\, b}\; L\bigl(w, b \,\big|\, \{x_i, y_i\}_{i \in [k]}\bigr). $$

The notation $L(w, b \mid \{x_i, y_i\})$ indicates that **$w$ and $b$ are the unknowns** of $L$, while the training set acts as **constant data**: once collected, it does not change during training. The relation $\in \operatorname*{argmin}$ (rather than $=$) accounts for the possibility that more than one pair $(w, b)$ achieves the minimum loss; in that case any such pair is admissible.

We do not constrain the domain of $(w, b)$: in principle, any real values are allowed. The training problem is to identify one such admissible pair.

## Global and local minima

Abstracting from the training problem, consider the minimisation of a generic function $f : \mathbb{R}^n \to \mathbb{R}$. We seek $x^* \in \mathbb{R}^n$ such that

$$ f(x^*) \;\leq\; f(x) \qquad \forall\, x \in \mathbb{R}^n. $$

Such an $x^*$ is called a **global minimum** of $f$. Finding global minima is hard in general, and for many functions impossible in reasonable time.

We often settle for a **local minimum**: a point $x^*$ that minimises $f$ on some *neighbourhood*,

$$ f(x^*) \;\leq\; f(x) \qquad \forall\, x \in B(x^*), $$

where $B(x^*) = \{x : \|x - x^*\|_2 \leq r\}$ is a Euclidean ball around $x^*$. Local minima are weaker than global ones — in NN training they typically yield a larger loss — but they are considerably easier to find.

On a 1-D function with several valleys, each valley bottom is a local minimum and the deepest one is the global minimum. Gradient methods — the class of algorithms studied below — only have access to local information about $f$, and cannot in general distinguish a local from a global minimum.

## Gradient descent (Cauchy, 1847)

Mathematical optimisation is a vast field, with specialised algorithms for many specialised problem classes. For training neural networks, the standard tool is one of the simplest methods available: **gradient descent** (GD), introduced by Augustin-Louis Cauchy in 1847.

Start from a point $x^{(0)} \in \mathbb{R}^n$ (possibly chosen at random). At each step, move in the direction along which $f$ decreases most rapidly. Calculus identifies this direction via the **gradient**

$$ \nabla f(x) \;=\; \begin{pmatrix} \partial f / \partial x_1 \\ \partial f / \partial x_2 \\ \vdots \\ \partial f / \partial x_n \end{pmatrix} \;\in\; \mathbb{R}^n. $$

Two properties of $\nabla f(x)$:

1. **Direction.** $\nabla f(x)$ points in the direction of **steepest ascent** of $f$ at $x$ — the direction along which $f$ grows fastest.
2. **Magnitude.** $\|\nabla f(x)\|_2$ measures the steepness of that ascent. At a minimum, the gradient vanishes: $\nabla f(x^*) = 0$.

Since the objective is to *minimise* $f$, we move in the opposite direction — along the **antigradient**, $-\nabla f(x)$. The update rule is

$$ \boxed{\; x^{(t+1)} \;:=\; x^{(t)} \;-\; \alpha \, \nabla f\bigl(x^{(t)}\bigr). \;} $$

This is the complete algorithm. The scalar $\alpha > 0$ is the **step length**; in the machine learning literature, it is called the **learning rate**.

> **Line search.** A more principled approach is to choose $\alpha$ adaptively at each step, selecting the value that maximises the decrease of $f$ along the antigradient direction. This procedure is called **line search**. It is rarely used in NN training: it is computationally expensive, and a small fixed $\alpha$ is usually sufficient in practice.

## What can go wrong — and why convexity matters

Two important caveats are worth noting from the outset:

- **GD does not always converge.** If $\alpha$ is too large, the iterates can overshoot the minimum and either oscillate or **diverge** — we observe this explicitly in §1 below. If $\alpha$ is too small, the iterates progress very slowly.
- **GD converges to a *local* minimum**, not necessarily a global one. The starting point $x^{(0)}$ matters: different initialisations can lead to different limiting points.

There is one important class of functions for which the second issue does not arise: **convex** functions. A function $f$ is convex when the line segment joining any two points on its graph lies above (or on) the graph. Algebraically:

$$ f\bigl(\lambda x_1 + (1-\lambda)\,x_2\bigr) \;\leq\; \lambda\, f(x_1) + (1-\lambda)\, f(x_2) \qquad \forall\, x_1, x_2,\; \lambda \in [0, 1]. $$

Informally, a convex function is **bowl-shaped**. For convex $f$, every local minimum is also a global minimum, and GD converges to it for any sufficiently small $\alpha$, independently of the starting point. This holds for the linear-regression loss in §2, which is why we will be able to compare GD's final iterate to a closed-form solution and observe agreement.

However: **most NN loss functions are not convex**. Even when the loss itself is convex (e.g. MSE), composing it with a piecewise-affine NN function — the typical form of a NN's input-output map — yields a non-convex loss with multiple local minima. Non-convex losses, and the optimisation challenges they introduce, are the subject of a later lecture.

In [ ]:
# Imports

import matplotlib.pyplot as plt
import numpy as np

## §1 — Univariate warm-up: GD on $f(x) = (x-3)^2$

Before tackling a multi-parameter loss, we apply GD to the simplest possible function: a 1-D parabola with a single global minimum at $x = 3$.

$$ f(x) = (x - 3)^2 \qquad \Longrightarrow \qquad f'(x) = 2(x - 3). $$

(Derive $f'$ on paper using the chain rule on $(x-3)^2$.)

The GD update for a univariate function is

$$ x^{(t+1)} := x^{(t)} - \alpha \, f'(x^{(t)}) = x^{(t)} - 2\alpha\,(x^{(t)} - 3). $$

The gradient vanishes at $x = 3$, so the iterates remain stationary once they reach it.

In [ ]:
# Plot f(x) and f'(x) on [-2, 8].
x = np.linspace(-2, 8, 200)
f = (x - 3) ** 2
f_prime = 2 * (x - 3)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(x, f, color="tab:blue", lw=2, label=r"$f(x)=(x-3)^2$")
ax1.set_xlabel(r"$x$")
ax1.set_ylabel(r"$f(x)$", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.axvline(3, color="gray", ls=":", lw=1, label=r"global min $x=3$")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(x, f_prime, color="tab:orange", lw=2, label=r"$f'(x)=2(x-3)$")
ax2.axhline(0, color="tab:orange", ls=":", lw=1)
ax2.set_ylabel(r"$f'(x)$", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

ax1.set_title("A parabola and its derivative")
fig.tight_layout()
plt.show()

### Exercise 1 — implement `gd_1d`

Write a function that runs $n$ iterations of GD on $f(x)=(x-3)^2$ from a starting point $x_0$ with learning rate $\alpha$, and returns the trajectories `xs` and `fs` (both of length `n_iters + 1`, including the starting point).

Implement it as a plain for-loop, without autograd.

```python
def gd_1d(x0, alpha, n_iters):
    """Returns xs (length n_iters+1) and fs (length n_iters+1)."""
    ...
```

In [ ]:
# TODO: implement gd_1d.
#   Hand-derived: f'(x) = 2 * (x - 3).
#   Need: lists xs and fs of length n_iters+1, INCLUDING the initial x0 and f(x0).
#   Use a plain for-loop and the manual update rule x <- x - alpha * f'(x).

def gd_1d(x0, alpha, n_iters):
    """GD on f(x) = (x - 3)**2. Returns lists xs and fs of length n_iters+1."""
    pass

We run **three reference cases** and compare:

| Case | $x_0$ | $\alpha$ | Expected behaviour |
|---|---|---|---|
| Smooth | 0.0 | 0.05 | Monotone descent toward $x=3$ in roughly a dozen steps. |
| Oscillating | 0.0 | 0.95 | Oscillation around $x=3$, but with eventual convergence. |
| Diverging | 0.0 | 1.05 | Divergence: $|x|$ grows without bound. |

(Predict the threshold for divergence on paper before running. Hint: the GD update can be rewritten as $x^{(t+1)} - 3 = (1 - 2\alpha)\,(x^{(t)} - 3)$. For which $\alpha$ does $|1 - 2\alpha| \geq 1$?)

In [ ]:
n_iters = 15
xs_smooth, fs_smooth   = gd_1d(0.0, alpha=0.05, n_iters=n_iters)
xs_oscill, fs_oscill   = gd_1d(0.0, alpha=0.95, n_iters=n_iters)
xs_diverg, fs_diverg   = gd_1d(0.0, alpha=1.05, n_iters=n_iters)

print(f"smooth:    final x = {xs_smooth[-1]:.6f},  f = {fs_smooth[-1]:.2e}")
print(f"oscill.:   final x = {xs_oscill[-1]:.6f},  f = {fs_oscill[-1]:.2e}")
print(f"divergent: final x = {xs_diverg[-1]:.6f},  f = {fs_diverg[-1]:.2e}")

In [ ]:
# Plot the iterates as red dots overlaid on the parabola, for each of the 3 runs.
x_grid = np.linspace(-5, 11, 400)
f_grid = (x_grid - 3) ** 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (xs, fs, title) in zip(
    axes,
    [
        (xs_smooth, fs_smooth, r"$\alpha=0.05$ — smooth"),
        (xs_oscill, fs_oscill, r"$\alpha=0.95$ — oscillating"),
        (xs_diverg, fs_diverg, r"$\alpha=1.05$ — diverging"),
    ],
):
    ax.plot(x_grid, f_grid, color="tab:blue", lw=1.5, label=r"$f(x)$")
    ax.plot(xs, fs, "o-", color="tab:red", ms=5, lw=0.8, label="GD iterates")
    ax.axvline(3, color="gray", ls=":", lw=1)
    ax.set_title(title)
    ax.set_xlabel(r"$x$")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper center")
axes[0].set_ylabel(r"$f(x)$")
fig.tight_layout()
plt.show()

In [ ]:
# Loss vs iteration for the three runs.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (fs, title) in zip(
    axes,
    [
        (fs_smooth, r"$\alpha=0.05$ — smooth"),
        (fs_oscill, r"$\alpha=0.95$ — oscillating"),
        (fs_diverg, r"$\alpha=1.05$ — diverging"),
    ],
):
    ax.plot(fs, "o-", color="tab:red", ms=4, lw=1)
    ax.set_title(title)
    ax.set_xlabel("iteration $t$")
    ax.grid(alpha=0.3)
axes[0].set_ylabel(r"$f(x^{(t)})$")
axes[2].set_yscale("log")            # the divergent run grows fast — log scale helps
fig.tight_layout()
plt.show()

## §2 — A two-parameter loss: linear regression

We now turn to a proper learning problem. Assume a hidden process generates data $(x_i, y_i)$, $i \in [k]$, via an affine rule corrupted by Gaussian noise:

$$ y_i \;=\; a^{\star} x_i \;+\; b^{\star} \;+\; \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2). $$

The unknowns are the two parameters $(a^{\star}, b^{\star})$ — the slope and intercept of the underlying line — to be learned from the noisy samples.

Following the **statistical-learning** approach, we choose a model family $f_{a,b}(x) = a x + b$ (the **inductive bias**) and the classical **mean squared error** as loss function:

$$ L(a, b) \;=\; \frac{1}{k}\sum_{i \in [k]} \bigl(\,a x_i + b - y_i\,\bigr)^2. $$

This is a function of two real parameters; its graph lives in $\mathbb{R}^3$ and can be plotted directly.

We restrict the problem to two parameters because the loss surface and the trajectory of iterates can then be visualised on a 2-D contour map. Real models have millions of parameters, and no such direct visualisation is possible.

Connecting back to the recap: this loss is a sum of squared linear-affine terms, hence **convex** (visibly a bowl in the 3-D plot below). The global minimum is therefore the only minimum any descent procedure can converge to.

### Exercise 2 — implement `make_data` and `loss`

Write two functions:

```python
def make_data(k, a_star, b_star, noise_std, x_low=-3.0, x_high=3.0, seed=0):
    """Returns (X, Y), both 1-D numpy arrays of length k."""
    ...

def loss(a, b, X, Y):
    """Returns the scalar MSE."""
    ...
```

Use `np.random.default_rng(seed)` to ensure reproducibility. After this exercise, `loss(a, b, X, Y)` is a plain numpy function: no autograd, no computational graph, only arithmetic on arrays.

In [ ]:
# TODO: implement make_data and loss.
#
# make_data(k, a_star, b_star, noise_std, x_low=-3.0, x_high=3.0, seed=0):
#     Need: (X, Y) — both 1-D numpy arrays of length k.
#     X is uniform in [x_low, x_high]; Y = a_star * X + b_star + N(0, noise_std).
#     Use np.random.default_rng(seed) for reproducibility.
#
# loss(a, b, X, Y):
#     Need: a scalar — the MSE (1/k) * sum((a*X + b - Y)**2).

def make_data(k, a_star, b_star, noise_std, x_low=-3.0, x_high=3.0, seed=0):
    pass


def loss(a, b, X, Y):
    pass

In [ ]:
# Generate the dataset and look at it.
k          = 100
a_star     = 2.0
b_star     = 0.5
noise_std  = 0.5
X, Y = make_data(k, a_star, b_star, noise_std, seed=0)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X, Y, s=25, color="tab:blue", alpha=0.7, edgecolor="k", lw=0.4, label="data")
x_line = np.linspace(X.min(), X.max(), 100)
ax.plot(x_line, a_star * x_line + b_star, "r--", lw=1.5,
        label=fr"true line: $y = {a_star}x + {b_star}$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title(f"Noisy linear data ($k={k}$, noise std = {noise_std})")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"L(a*, b*)  = {loss(a_star, b_star, X, Y):.4f}")
print(f"L(0,  0)   = {loss(0.0, 0.0, X, Y):.4f}    # baseline, no fit")

In [ ]:
# Visualize the loss surface L(a, b) — both as a 3-D bowl and as a 2-D contour.
a_grid = np.linspace(0.0, 4.0, 80)
b_grid = np.linspace(-2.0, 3.0, 80)
A, B = np.meshgrid(a_grid, b_grid)
L = np.array([[loss(a, b, X, Y) for a in a_grid] for b in b_grid])

fig = plt.figure(figsize=(13, 5))

ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(A, B, L, cmap="viridis", alpha=0.85, edgecolor="none")
ax3d.set_xlabel(r"$a$")
ax3d.set_ylabel(r"$b$")
ax3d.set_zlabel(r"$L(a, b)$")
ax3d.set_title("Loss surface (a paraboloid bowl)")

ax2d = fig.add_subplot(1, 2, 2)
cs = ax2d.contourf(A, B, L, levels=25, cmap="viridis")
ax2d.contour(A, B, L, levels=15, colors="white", linewidths=0.5, alpha=0.5)
ax2d.scatter([a_star], [b_star], marker="*", s=200, c="red",
             edgecolor="white", lw=0.8, label=fr"$(a^\star, b^\star)=({a_star}, {b_star})$")
ax2d.set_xlabel(r"$a$")
ax2d.set_ylabel(r"$b$")
ax2d.set_title("Loss contour")
ax2d.legend(loc="upper right")
fig.colorbar(cs, ax=ax2d, label="$L(a, b)$")
plt.tight_layout()
plt.show()

## §3 — Gradient descent on $L(a, b)$

We now derive the gradient of $L$ by hand, implement it in numpy, and run GD.

Let $L_i(a,b) := (a x_i + b - y_i)^2$ denote the per-sample contribution, so that $L = \frac{1}{k}\sum_i L_i$. Since differentiation is linear,

$$ \frac{\partial L}{\partial a} \;=\; \frac{1}{k}\sum_i \frac{\partial L_i}{\partial a}, \qquad \frac{\partial L}{\partial b} \;=\; \frac{1}{k}\sum_i \frac{\partial L_i}{\partial b}. $$

For a single sample, applying the chain rule to $L_i = (a x_i + b - y_i)^2$ with $r_i := a x_i + b - y_i$:

$$ \frac{\partial L_i}{\partial a} \;=\; 2 r_i \cdot \frac{\partial r_i}{\partial a} \;=\; 2 r_i \, x_i, \qquad \frac{\partial L_i}{\partial b} \;=\; 2 r_i \cdot \frac{\partial r_i}{\partial b} \;=\; 2 r_i. $$

Summing and dividing by $k$:

$$ \boxed{\;\frac{\partial L}{\partial a} \;=\; \frac{2}{k}\sum_i x_i\,(a x_i + b - y_i), \qquad \frac{\partial L}{\partial b} \;=\; \frac{2}{k}\sum_i (a x_i + b - y_i).\;} $$

This completes the derivation. No autograd, no computational graph; only calculus. The gradient is

$$ \nabla L(a, b) = \begin{pmatrix} \partial L / \partial a \\ \partial L / \partial b \end{pmatrix} \in \mathbb{R}^2, $$

and the GD update is

$$ \begin{pmatrix} a \\ b \end{pmatrix}^{(t+1)} \;:=\; \begin{pmatrix} a \\ b \end{pmatrix}^{(t)} - \alpha \, \nabla L\bigl(a^{(t)}, b^{(t)}\bigr). $$

### Exercise 3 — implement `grad_L`

Translate the boxed formulas above into a function:

```python
def grad_L(a, b, X, Y):
    """Returns (dL_da, dL_db) — two scalars."""
    ...
```

*Hint*: compute the residual `r = a*X + b - Y` once (a vector) and reuse it for both partials. The implementation is then two lines.

In [ ]:
# TODO: implement grad_L.
#   Need: a tuple (dL_da, dL_db) of two scalars.
#   Translate the hand derivation directly:
#     dL/da = (2/k) * sum(x_i * (a*x_i + b - y_i))
#     dL/db = (2/k) * sum(a*x_i + b - y_i)
#   Hint: compute the residual r = a*X + b - Y once, then reuse.

def grad_L(a, b, X, Y):
    """Analytic gradient of L w.r.t. (a, b). Returns (dL_da, dL_db)."""
    pass

**Sanity check.** Whenever a gradient is derived by hand, it should be verified numerically. A standard technique is **central differences**:

$$ \frac{\partial L}{\partial a}(a, b) \;\approx\; \frac{L(a + \varepsilon, b) - L(a - \varepsilon, b)}{2\varepsilon} $$

for a small $\varepsilon$. If `grad_L` agrees with the numerical gradient to roughly 6 decimal places, the hand derivation is almost certainly correct.

In [ ]:
def numerical_grad(f, a, b, eps=1e-5):
    """Central differences of a scalar function f(a, b)."""
    da = (f(a + eps, b) - f(a - eps, b)) / (2 * eps)
    db = (f(a, b + eps) - f(a, b - eps)) / (2 * eps)
    return da, db

rng = np.random.default_rng(0)
for _ in range(3):
    a_test, b_test = rng.uniform(-2, 4), rng.uniform(-2, 4)
    g_hand = grad_L(a_test, b_test, X, Y)
    g_num  = numerical_grad(lambda a, b: loss(a, b, X, Y), a_test, b_test)
    print(f"(a, b)=({a_test:+.3f}, {b_test:+.3f})  "
          f"hand=({g_hand[0]:+.6f}, {g_hand[1]:+.6f})  "
          f"num=({g_num[0]:+.6f}, {g_num[1]:+.6f})  "
          f"|diff|={max(abs(g_hand[0]-g_num[0]), abs(g_hand[1]-g_num[1])):.2e}")

**An exact reference solution.** For linear regression, the minimum of $L(a,b)$ has a closed form — the **ordinary least squares** (OLS) solution:

$$ a_{\rm OLS} \;=\; \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2}, \qquad b_{\rm OLS} \;=\; \bar y - a_{\rm OLS}\,\bar x. $$

We compute it directly and use it as a reference against which to check GD's iterate. Such a reference is available only because the loss is convex; for non-convex losses (e.g. those arising in deep neural networks) no closed form exists.

In [ ]:
X_centered = X - X.mean()
Y_centered = Y - Y.mean()
a_ols = (X_centered * Y_centered).sum() / (X_centered ** 2).sum()
b_ols = Y.mean() - a_ols * X.mean()

print(f"OLS solution:    a_ols = {a_ols:.6f},   b_ols = {b_ols:.6f}")
print(f"Ground truth:    a*    = {a_star:.6f},   b*    = {b_star:.6f}")
print(f"L(a_ols, b_ols)  = {loss(a_ols, b_ols, X, Y):.6f}")
print(f"L(a*,    b*)    = {loss(a_star, b_star, X, Y):.6f}")
print()
print("Note: OLS != truth because of finite-sample noise. OLS minimises the sample loss;")
print("(a*, b*) minimises the population loss. The two coincide only as k -> infinity.")

### Exercise 4 — implement `gd_2d`

```python
def gd_2d(a0, b0, alpha, n_iters, X, Y):
    """Returns trajectory of shape (n_iters+1, 2) and losses of shape (n_iters+1,)."""
    ...
```

Implement it as a plain for-loop. At each iteration: compute `grad_L`, take a step, and store the new `(a, b)` and `loss(a, b, X, Y)`.

In [ ]:
# TODO: implement gd_2d.
#   Need: traj (shape (n_iters+1, 2)) and losses (shape (n_iters+1,)) — numpy arrays.
#   Initialize with (a0, b0) and the corresponding loss; then run n_iters manual updates:
#     (a, b) <- (a, b) - alpha * (dL/da, dL/db)
#   Append both the new (a, b) and loss(a, b, X, Y) at every iteration.

def gd_2d(a0, b0, alpha, n_iters, X, Y):
    """Plain GD on the 2-parameter MSE loss."""
    pass

In [ ]:
# Run three flagship cases. All start at (a, b) = (0, 0), 50 iterations.
n_iters = 50
traj_good,    losses_good    = gd_2d(0.0, 0.0, alpha=0.05,  n_iters=n_iters, X=X, Y=Y)
traj_glacial, losses_glacial = gd_2d(0.0, 0.0, alpha=0.005, n_iters=n_iters, X=X, Y=Y)
traj_diverg,  losses_diverg  = gd_2d(0.0, 0.0, alpha=0.5,   n_iters=n_iters, X=X, Y=Y)

for name, traj, losses in [
    ("alpha=0.05  (good)   ", traj_good,    losses_good),
    ("alpha=0.005 (glacial)", traj_glacial, losses_glacial),
    ("alpha=0.5   (diverg) ", traj_diverg,  losses_diverg),
]:
    a, b = traj[-1]
    print(f"{name}  final (a, b) = ({a:+.4f}, {b:+.4f})   L = {losses[-1]:.4e}")

print()
print(f"OLS reference        final (a, b) = ({a_ols:+.4f}, {b_ols:+.4f})")

In [ ]:
# A reusable helper to draw the loss contour (we'll call this in §3 and §4).
def plot_contour(ax, X, Y, a_range=(-0.5, 4.0), b_range=(-2.0, 3.0), n=80):
    a_grid = np.linspace(*a_range, n)
    b_grid = np.linspace(*b_range, n)
    A, B = np.meshgrid(a_grid, b_grid)
    L_grid = np.array([[loss(a, b, X, Y) for a in a_grid] for b in b_grid])
    cs = ax.contourf(A, B, L_grid, levels=25, cmap="viridis")
    ax.contour(A, B, L_grid, levels=15, colors="white", linewidths=0.4, alpha=0.4)
    ax.scatter([a_ols], [b_ols], marker="*", s=180, c="red",
               edgecolor="white", lw=0.7, zorder=5, label="OLS")
    ax.set_xlabel(r"$a$")
    ax.set_ylabel(r"$b$")
    return cs

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (traj, title) in zip(
    axes,
    [
        (traj_good,    r"$\alpha=0.05$ — good"),
        (traj_glacial, r"$\alpha=0.005$ — glacial"),
        (traj_diverg,  r"$\alpha=0.5$ — diverging"),
    ],
):
    plot_contour(ax, X, Y)
    ax.plot(traj[:, 0], traj[:, 1], "o-", color="red", ms=4, lw=1, label="GD path")
    ax.scatter(traj[0, 0], traj[0, 1], marker="s", s=80, color="black", zorder=6, label="start")
    ax.set_title(title)
    ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
# Loss-vs-iteration for the three runs.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (losses, title) in zip(
    axes,
    [
        (losses_good,    r"$\alpha=0.05$ — good"),
        (losses_glacial, r"$\alpha=0.005$ — glacial"),
        (losses_diverg,  r"$\alpha=0.5$ — diverging"),
    ],
):
    ax.plot(losses, "o-", ms=3, lw=1, color="tab:red")
    ax.axhline(loss(a_ols, b_ols, X, Y), color="black", ls=":", lw=1, label="OLS loss")
    ax.set_yscale("log")
    ax.set_xlabel("iteration $t$")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel(r"$L(a^{(t)}, b^{(t)})$")
fig.tight_layout()
plt.show()

### Bonus — verify the gradient symbolically with `sympy`

We have already verified the hand-derived gradient *numerically* via central differences. For additional confidence, the gradient can also be verified **symbolically** with `sympy`: compute the partial derivatives of $L_i$ with respect to $a$ and $b$ as algebraic expressions, and check whether they match the boxed formulas.

This step is **optional** — the hand derivation together with the numerical check is already sufficient. However, when deriving more involved gradients, symbolic verification with `sympy` is a useful safeguard against algebraic errors.

(The cell below is import-guarded: if `sympy` is not installed, it prints a message and skips. `sympy` is preinstalled on Colab; for local installations, run `pip install sympy`.)

In [ ]:
try:
    import sympy as sp
except ImportError:
    print("sympy not installed — skipping bonus.")
else:
    a_s, b_s, x_s, y_s = sp.symbols("a b x y", real=True)
    L_i = (a_s * x_s + b_s - y_s) ** 2
    print("L_i        =", L_i)
    print("dL_i/da    =", sp.factor(sp.diff(L_i, a_s)))
    print("dL_i/db    =", sp.factor(sp.diff(L_i, b_s)))
    print()
    print("Expected (from our derivation):")
    print("  dL_i/da  =  2*x*(a*x + b - y)")
    print("  dL_i/db  =  2*(a*x + b - y)")

## §4 — Stochastic Gradient Descent

Plain GD has a serious practical limitation: it computes $\nabla L = \sum_i \nabla L_i$ over the entire training set at every step. This is acceptable for $k=100$, but in deep learning $k$ can be in the millions or billions, in which case iterating over the full dataset at every step is computationally prohibitive.

**Stochastic Gradient Descent** (SGD, Robbins & Monro 1951) replaces the full-data gradient with an **approximation** computed on a random subset $B^{(t)} \subset [k]$ — a **minibatch**:

$$ \begin{pmatrix} a \\ b \end{pmatrix}^{(t+1)} \;:=\; \begin{pmatrix} a \\ b \end{pmatrix}^{(t)} \;-\; \alpha \sum_{i \in B^{(t)}} \nabla L_i\bigl(a^{(t)}, b^{(t)}\bigr). $$

(Note: the gradients of the samples in the batch are *summed*, with no $1/|B|$ factor. This is the classical convention used in the lecture; with a $1/|B|$ factor in the loss, the difference is absorbed into $\alpha$.)

Two consequences:

1. **Inexpensive iterations.** A step costs $O(|B|)$ rather than $O(k)$.
2. **Noisy trajectory.** The batch gradient is an *estimator* of the full gradient; individual estimates can point in directions that locally increase the loss. This noise is not always undesirable: in non-convex problems, it can allow SGD to escape poor basins of attraction in which plain GD would remain. For our convex linear-regression loss this aspect is less prominent, but it is central in the non-convex setting.

Terminology: a single pass over the entire training set is one **epoch**. With $k=100$ and $|B|=10$, an epoch consists of 10 SGD iterations; with $|B|=1$, of 100.

### Exercise 5 — implement `sgd_2d`

```python
def sgd_2d(a0, b0, alpha, n_iters, batch_size, X, Y, seed=0):
    """Returns trajectory (n_iters+1, 2) and losses (n_iters+1,)."""
    ...
```

Same shape contract as `gd_2d`, plus a `batch_size` argument. Within each iteration:

1. Sample a batch of `batch_size` indices **without replacement**.
2. Reshuffle the index pool when it runs out (epoch boundary).
3. Compute `grad_L` on the **batch only**.
4. Take a step.
5. Append the new `(a, b)` and the **full-data** `loss(a, b, X, Y)`, so that the loss curve is directly comparable with `gd_2d`.

*Hint*: use `rng = np.random.default_rng(seed)` and `rng.permutation(k)` to manage the shuffling.

In [ ]:
# TODO: implement sgd_2d.
#   Same shape contract as gd_2d, plus a `batch_size` argument and an internal RNG.
#   At each iteration:
#     - sample a batch of indices of size batch_size, without replacement
#     - reshuffle when the current permutation is exhausted (epoch boundary)
#     - compute grad_L on the batch only
#     - take a step
#     - append the new (a, b) and the FULL-DATA loss(a, b, X, Y) so curves
#       are directly comparable with gd_2d's losses.
#   Hint: np.random.default_rng(seed); use rng.permutation(k) for shuffling.

def sgd_2d(a0, b0, alpha, n_iters, batch_size, X, Y, seed=0):
    pass

In [ ]:
# Run GD and two SGD variants from the same starting point and the same alpha.
n_iters    = 80
alpha      = 0.05
a0, b0     = 0.0, 0.0

traj_gd,    losses_gd    = gd_2d(a0, b0, alpha, n_iters, X, Y)
traj_sgd10, losses_sgd10 = sgd_2d(a0, b0, alpha, n_iters, batch_size=10, X=X, Y=Y, seed=0)
traj_sgd1,  losses_sgd1  = sgd_2d(a0, b0, alpha, n_iters, batch_size=1,  X=X, Y=Y, seed=0)

for name, traj, losses in [
    ("GD     (full batch) ", traj_gd,    losses_gd),
    ("SGD     batch=10    ", traj_sgd10, losses_sgd10),
    ("SGD     batch= 1    ", traj_sgd1,  losses_sgd1),
]:
    a, b = traj[-1]
    print(f"{name}  final (a, b) = ({a:+.4f}, {b:+.4f})   L = {losses[-1]:.4e}")

print()
print(f"OLS reference        final (a, b) = ({a_ols:+.4f}, {b_ols:+.4f})")

In [ ]:
# Side-by-side trajectories on the contour map.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (traj, title, color) in zip(
    axes,
    [
        (traj_gd,    "GD (full batch)",    "red"),
        (traj_sgd10, "SGD, batch = 10",    "orange"),
        (traj_sgd1,  "SGD, batch = 1",     "magenta"),
    ],
):
    plot_contour(ax, X, Y)
    ax.plot(traj[:, 0], traj[:, 1], "o-", color=color, ms=3, lw=0.8, label=title)
    ax.scatter(traj[0, 0], traj[0, 1], marker="s", s=80, color="black", zorder=6)
    ax.set_title(title)
    ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
# Loss vs iteration — same alpha, same starting point.
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (losses, title, color) in zip(
    axes,
    [
        (losses_gd,    "GD (full batch)",  "red"),
        (losses_sgd10, "SGD, batch = 10",  "orange"),
        (losses_sgd1,  "SGD, batch = 1",   "magenta"),
    ],
):
    ax.plot(losses, "o-", ms=3, lw=0.8, color=color)
    ax.axhline(loss(a_ols, b_ols, X, Y), color="black", ls=":", lw=1, label="OLS loss")
    ax.set_yscale("log")
    ax.set_xlabel("iteration $t$")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel(r"$L(a^{(t)}, b^{(t)})$ (full data)")
fig.tight_layout()
plt.show()

**Expected behaviour.**

- **GD** produces a smooth curve descending monotonically toward the OLS minimum.
- **SGD with batch = 10** also reaches the minimum, but along a visibly noisier path — individual steps may not decrease the loss.
- **SGD with batch = 1** is highly noisy. Individual steps may move *uphill* in the full-data loss. This is consistent with the theory: the batch gradient is an unbiased estimator of the full gradient, but a single sample is a high-variance estimate.

Vary the `seed` argument of `sgd_2d` and re-run: the trajectory differs across runs — this is the *stochastic* component of SGD.

## §5 — Homework: a learning-rate schedule

Sections §1 and §3 showed that the choice of $\alpha$ is delicate: small $\alpha$ leads to slow convergence, large $\alpha$ to divergence. A common remedy is to **decrease $\alpha$ over time** — larger steps early on (for exploration) and smaller steps later (for refinement).

The simplest schedule is **step decay**: keep $\alpha$ constant for $s$ iterations, then multiply it by a factor $\gamma \in (0, 1)$.

$$ \alpha_t \;=\; \alpha_0 \cdot \gamma^{\lfloor t / s \rfloor}. $$

**Task.** Modify `sgd_2d` to accept an `lr_schedule` callable in place of a fixed `alpha`, and re-run the noisy `batch=1` case from §4 with $\alpha_0 = 0.1$, $\gamma = 0.5$, $s = 10$. Does the loss curve stabilise more cleanly than with a fixed $\alpha$?

Plot $\alpha_t$ against $t$ alongside the loss curve to visualise the schedule.